# Random Forest Regression — Google Colab

**Goal:** Predict a continuous target using **Random Forest Regression** — an **ensemble** of Decision Trees that reduces overfitting and improves accuracy.

| Example | Feature(s) (X) | Target (y) | Dataset |
|---------|----------------|------------|---------|
| **Example 1** | Level | Salary | `../Datasets/Position_Salaries.csv` |
| **Example 2** | Area_sqft, Bedrooms, Age_years | Price | `../Datasets/house_price.csv` |

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Install & Imports |
| — | Algorithm Guide | Bagging, bootstrap, feature randomness |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

---
# Algorithm Guide — Random Forest Regression

## What is Random Forest?

**Random Forest** = **many Decision Trees** trained on **different random subsets** of data and features, then their predictions are **averaged**.

It is a **Bagging** (Bootstrap Aggregating) ensemble method invented by Leo Breiman (2001).

## How It Works (Step by Step)

| Step | Name | What Happens |
|------|------|--------------|
| 1 | **Bootstrap sample** | Draw n rows **with replacement** from training data |
| 2 | **Random features** | At each split, consider only a **random subset** of features |
| 3 | **Grow tree** | Build a full CART tree on that bootstrap sample |
| 4 | **Repeat** | Create `n_estimators` trees (e.g. 100) |
| 5 | **Aggregate** | Final prediction = **mean** of all tree predictions |

## Prediction Rule (Regression)

For a new sample **x**:

`ŷ = (1/B) · Σ ŷ_b(x)`

where **B** = number of trees (`n_estimators`) and **ŷ_b** = prediction from tree **b**.

## Key Hyperparameters

| Parameter | Role | Effect |
|-----------|------|--------|
| `n_estimators` | Number of trees in the forest | More trees → more stable (diminishing returns) |
| `max_depth` | Max depth of each tree | Limits overfitting per tree |
| `min_samples_split` | Min samples to split a node | Higher → simpler trees |
| `min_samples_leaf` | Min samples in a leaf | Higher → smoother predictions |
| `max_features` | Features considered per split | `'sqrt'` or `'1.0'` — adds diversity between trees |
| `bootstrap` | Use bootstrap sampling | `True` (default) — core of bagging |
| `random_state` | Random seed | Reproducible forest |

## Random Forest vs Single Decision Tree (CART)

| | Single CART | Random Forest |
|---|-------------|---------------|
| Trees | **1** tree | **Many** trees (ensemble) |
| Overfitting | **High** risk if deep | **Lower** — averaging reduces variance |
| Prediction | Step function | **Smoother** average of steps |
| Interpretability | Easy (`plot_tree`) | Harder — use **feature importances** |
| Scaling | Not required | **Not required** |
| Speed | Fast | Slower (trains B trees) |

## Feature Importance

Random Forest computes importance by measuring how much each feature **reduces MSE** across all splits in all trees. Higher value = more influential for predictions.

## What the Student Must Remember

1. Random Forest = **Bagging + random feature subsets** at each split.
2. Final prediction = **average** of all tree outputs (regression).
3. **No feature scaling** needed — same as Decision Trees.
4. More trees (`n_estimators`) usually help until performance plateaus.
5. Use **feature importances** to understand which inputs matter most.

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
# Install required libraries quietly (-q hides output)
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for data handling, preprocessing, modeling, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, sklearn RandomForestRegressor, and metrics.

In [ ]:
# --- Import libraries ---
import numpy as np              # Numerical operations and arrays
import pandas as pd             # Load and manipulate tabular data
import matplotlib.pyplot as plt # Create charts and plots
import seaborn as sns           # Statistical visualizations (optional styling)

from sklearn.model_selection import train_test_split       # Split data into train/test
from sklearn.impute import SimpleImputer                   # Fill missing values
from sklearn.ensemble import RandomForestRegressor         # Random Forest ensemble regressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics

plt.rcParams['figure.figsize'] = (10, 6)  # Default plot size: width=10, height=6 inches
sns.set_theme(style='whitegrid')            # Clean white background with grid lines
np.random.seed(42)                          # Fix random seed for reproducible splits

print('Libraries ready')                      # Confirm all imports loaded successfully

---
# Example 1: Position Salaries — Random Forest Regression

Predict **Salary** from job **Level**. The relationship is **non-linear** — Random Forest averages many trees for smoother predictions than a single CART.

| Column | Role | Description |
|--------|------|-------------|
| `Level` | Feature (X) | Job level (1–10) |
| `Salary` | Target (y) | Annual salary in USD |

**File:** `../Datasets/Position_Salaries.csv`

---
# Phase 1: Data Pre-processing

Prepare the data before training — same template is reused for other algorithms.

## Example 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `../Datasets/Position_Salaries.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('../Datasets/Position_Salaries.csv')  # Read CSV into a DataFrame

FEATURE_COL = 'Level'   # Independent variable (X) — job level
TARGET_COL = 'Salary'   # Dependent variable (y) — salary to predict

print('First 5 rows:')          # Print a label for the table below
display(dataset.head())         # Show the first 5 rows to inspect the data

print('\nDataset info:')       # Print a label for column types and null counts
dataset.info()                  # Show column names, data types, and non-null counts

print('\nStatistical summary:')  # Print a label for numeric statistics
display(dataset.describe())       # Show count, mean, std, min, max, quartiles

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')  # Total rows and columns

## Example 1 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values, remove duplicates, and apply imputation if needed.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')  # Print a label
print(dataset.isnull().sum())        # Count NaN values in each column

rows_before = len(dataset)                              # Store row count before cleaning
dataset = dataset.drop_duplicates().reset_index(drop=True)  # Remove duplicate rows
rows_after = len(dataset)                               # Store row count after deduplication
print(f'\nDuplicates removed: {rows_before - rows_after}')  # Show duplicate count

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()  # Numeric columns only
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # Fill NaN with column mean
if dataset.isnull().sum().sum() > 0:               # If missing values exist
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])  # Impute numeric columns
    print('Missing values imputed with mean')      # Confirm imputation
else:
    print('No missing values — imputer not applied')  # Skip when complete

print(f'\nRows after cleaning: {rows_after}')    # Final row count

## Example 1 — Cell 3: Categorical Data Encoding

We use `Level` (numeric). The `Position` column is text — skipped because Level already encodes rank.

**What this cell does:** Checks for categorical columns and encodes if needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()  # Find text columns
print(f'Categorical columns (not used as X): {cat_cols}')  # Position names — informational only
print(f'Feature used for Random Forest: {FEATURE_COL}')    # Level is numeric — no encoding needed
print('No encoding required — X is numeric.')

## Example 1 — Cell 4: Splitting the Data

Define X (Level) and y (Salary), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[[FEATURE_COL]].values  # Feature matrix: Level (2D array for sklearn)
y = dataset[TARGET_COL].values     # Target vector: Salary values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                  # Data to split
    test_size=0.2,         # 20% test, 80% train
    random_state=42        # Reproducible split
)

print(f'X_train shape: {X_train.shape}')  # Training features shape
print(f'X_test shape:  {X_test.shape}')   # Test features shape
print(f'y_train shape: {y_train.shape}')  # Training targets shape
print(f'y_test shape:  {y_test.shape}')   # Test targets shape

> **Note:** Random Forest Regression does **not** require feature scaling. Trees split on thresholds — scale does not matter.

---
# Phase 2: Random Forest Regression

Train a Random Forest to predict Salary from Level.

## Example 1 — Cell 5: Train the Model

Train `RandomForestRegressor` with `n_estimators=100` trees. Each tree sees a bootstrap sample.

**What this cell does:** Fits the forest and prints the number of trees.

In [ ]:
# Step 5) Train Random Forest Regressor

regressor = RandomForestRegressor(
    n_estimators=100,      # Number of trees in the forest
    max_depth=4,           # Limit depth per tree (small dataset)
    min_samples_leaf=1,    # Minimum samples required in a leaf node
    random_state=42,       # Reproducible bootstrap and splits
    n_jobs=-1              # Use all CPU cores for faster training
)

regressor.fit(X_train, y_train)  # Train all trees on bootstrap samples

print('Random Forest trained successfully.')              # Confirm training complete
print(f'Number of trees (estimators): {len(regressor.estimators_)}')  # Should equal n_estimators
print(f'Feature importances: {regressor.feature_importances_}')        # Importance per feature

## Example 1 — Cell 6: Predict

Predict Salary on the test set by averaging all tree predictions.

**What this cell does:** Generates ensemble predictions.

In [ ]:
# Step 6) Predict

y_pred_train = regressor.predict(X_train)  # Average tree predictions for training data
y_pred_test = regressor.predict(X_test)    # Average tree predictions for test data

print('Sample predictions (Test set):')  # Print a label
for i in range(len(y_test)):             # Show all test predictions
    print(f'  Level={X_test[i][0]:.0f} -> Actual=${y_test[i]:,.0f}, Predicted=${y_pred_test[i]:,.0f}')

## Example 1 — Cell 7: Visualization

Plot the **Random Forest prediction curve** (smoother than single CART) and compare with raw data.

**What this cell does:** Shows ensemble predictions vs actual salary levels.

In [ ]:
# Step 7) Visualization — RF curve + scatter

fig, ax = plt.subplots(figsize=(10, 6))  # Single plot for clarity

X_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)  # 500 points for smooth curve
y_plot = regressor.predict(X_plot)                            # RF predictions (averaged trees)

ax.scatter(X_train, y_train, color='blue', label='Training', s=80, zorder=3)   # Training points
ax.scatter(X_test, y_test, color='green', label='Test', s=80, zorder=3)       # Test points
ax.plot(X_plot, y_plot, color='red', linewidth=2, label='Random Forest prediction')  # Smoothed curve
ax.set_xlabel('Level')             # X-axis label
ax.set_ylabel('Salary (USD)')     # Y-axis label
ax.set_title('Random Forest — Position Salaries')  # Chart title
ax.legend()                        # Show legend

plt.tight_layout()  # Adjust spacing
plt.show()          # Display plot

## Example 1 — Cell 8: Evaluation

Evaluate Random Forest with MAE, RMSE, and R² on the test set.

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation
mae = mean_absolute_error(y_test, y_pred_test)              # Mean absolute error in USD
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))     # Root mean squared error
r2 = r2_score(y_test, y_pred_test)                          # Variance explained

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'Mean Absolute Error (USD)',
        'Root Mean Squared Error (USD)',
        'Coefficient of Determination (1 = perfect)'
    ]
})

display(results.round(4))  # Show metrics table
print(f'\nExample 1 Test R² = {r2:.4f}')  # Print R² summary

## Why does Random Forest work for Position Salaries?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Non-linear salary jumps** | Many trees capture different step patterns; averaging smooths predictions |
| 2 | **Reduces overfitting** | Bagging lowers variance compared to one deep CART on 10 rows |
| 3 | **No scaling needed** | Trees compare `Level ≤ threshold` — raw values work fine |
| 4 | **Bootstrap diversity** | Each tree sees a slightly different sample — robust ensemble |
| 5 | **Better generalization** | Usually outperforms a single tree on small noisy data |

> **Summary:** Random Forest **averages** many CART models — smoother and often more accurate than one tree alone.

## Understanding R² — Example 1 (Position Salaries)

**R² (Coefficient of Determination)** measures how much of the salary variance is explained by Level.

| R² Value | Meaning |
|----------|---------|
| **1.0** | Perfect predictions |
| **0.7–0.9** | Strong fit |
| **0.4–0.7** | Moderate fit (common with tiny datasets) |
| **0.0** | Model no better than predicting the mean |
| **< 0** | Worse than the mean |

> With only **10 rows** and **2 test samples**, R² can vary a lot — focus on the **pattern** of predictions, not only one number.

---
# Example 2: House Price — Random Forest Regression

Predict **Price** from three property features using Random Forest with **multiple features**.

| Column | Role | Description |
|--------|------|-------------|
| `Area_sqft` | Feature (X₁) | Living area in square feet |
| `Bedrooms` | Feature (X₂) | Number of bedrooms |
| `Age_years` | Feature (X₃) | Age of the house in years |
| `Price` | Target (y) | Sale price in USD |

**File:** `../Datasets/house_price.csv`

## Example 2 — Cell 1: Load and Explore Data

Load the house price CSV and inspect the data.

**What this cell does:** Reads `../Datasets/house_price.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('../Datasets/house_price.csv')  # Read CSV into a DataFrame

FEATURE_COLS = ['Area_sqft', 'Bedrooms', 'Age_years']  # Three input features
TARGET_COL = 'Price'                                    # Target variable

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 2 — Cell 2: Data Cleaning (Handling Missing Values)

This dataset includes missing values to demonstrate `SimpleImputer`.

**What this cell does:** Checks for nulls, removes duplicates, and imputes missing values.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[FEATURE_COLS + [TARGET_COL]] = imputer.fit_transform(dataset[FEATURE_COLS + [TARGET_COL]])
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Example 2 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')

## Example 2 — Cell 4: Splitting the Data

Define X (3 features) and y (Price), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[FEATURE_COLS].values  # Feature matrix: 3 columns
y = dataset[TARGET_COL].values    # Target vector: Price values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train shape: {X_train.shape}')  # (n_train, 3)
print(f'X_test shape:  {X_test.shape}')   # (n_test, 3)
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

## Example 2 — Cell 5: Train the Model

Train Random Forest with multiple features — each tree splits on random feature subsets.

**What this cell does:** Fits the forest and reports feature importances.

In [ ]:
# Step 5) Train Random Forest Regressor

regressor = RandomForestRegressor(
    n_estimators=100,      # 100 trees in the ensemble
    max_depth=8,           # Allow deeper trees — more data than Example 1
    min_samples_split=4,     # Need at least 4 samples to split a node
    min_samples_leaf=2,      # Each leaf must have at least 2 samples
    max_features=1.0,        # Use all 3 features (small feature set)
    random_state=42,
    n_jobs=-1                # Parallel training across CPU cores
)

regressor.fit(X_train, y_train)  # Train forest on all 3 features

print('Random Forest trained successfully.')
print(f'Number of trees: {len(regressor.estimators_)}')

print('\nFeature importances:')  # Averaged across all trees
for name, imp in zip(FEATURE_COLS, regressor.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')  # Higher = more important for predictions

## Example 2 — Cell 6: Predict

Predict house prices on the test set.

**What this cell does:** Generates ensemble predictions by averaging tree outputs.

In [ ]:
# Step 6) Predict

y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    print(f'  Actual=${y_test[i]:,.0f}, Predicted=${y_pred_test[i]:,.0f}')

## Example 2 — Cell 7: Visualization

Plot **Actual vs Predicted** and **feature importances**.

**What this cell does:** Visualizes model performance and which features matter most.

In [ ]:
# Step 7) Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # Two subplots

axes[0].scatter(y_test, y_pred_test, color='green', alpha=0.7)  # Actual vs predicted
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Predicted Price (USD)')
axes[0].set_title('Actual vs Predicted — House Price (Random Forest)')
axes[0].legend()

axes[1].barh(FEATURE_COLS, regressor.feature_importances_, color='darkgreen')  # Feature importance chart
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importances (Random Forest)')

plt.tight_layout()
plt.show()

## Example 2 — Cell 8: Evaluation

Evaluate Random Forest performance with MAE, RMSE, and R².

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'Mean Absolute Error (USD)',
        'Root Mean Squared Error (USD)',
        'Coefficient of Determination (1 = perfect)'
    ]
})

display(results.round(4))
print(f'\nExample 2 Test R² = {r2:.4f}')

## Why does Random Forest work well for House Price?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Multiple features** | Forest splits on Area, Bedrooms, and Age across many trees |
| 2 | **Non-linear interactions** | Trees capture e.g. large Area + many Bedrooms → higher Price |
| 3 | **Feature importances** | Shows which feature contributes most (averaged over all trees) |
| 4 | **No scaling needed** | Raw sqft, bedroom count, and age work directly |
| 5 | **Strong R²** | Ensemble averaging typically beats a single CART on this dataset |

> **Compare:** Random Forest vs CART — same interpretability trade-off, but RF usually gives **higher accuracy** and **lower overfitting**.

## Understanding R² — Example 2 (House Price)

**R²** tells us how well Area, Bedrooms, and Age together explain house prices.

| R² Value | Meaning |
|----------|---------|
| **> 0.85** | Excellent — model captures most price variation |
| **0.70–0.85** | Good fit for real-world property data |
| **< 0.50** | Weak — consider more features or different model |

> Random Forest often achieves **higher R²** than a single Decision Tree because averaging **reduces prediction error**.